# Raw data export: test example

Shows what `raw_data_export.py` produces for a small selection of species: for each species and each data type, the runs found in ENA, the representative runs chosen, and why each one was chosen.
The result is one JSON file for the whole selection, written next to this notebook (`example_raw_data_export.json`). No reads file is downloaded, only their links.

In [3]:
import sys
sys.path.append("../..")

import json
from raw_data.scripts_raw_data.raw_data_export import collect_raw_data, write_export

## 1. The selection

Each species is given by its taxid, with its name to make the file easier to read. Two bats are there because bats are one of the supervisors' test cases.
The last one is there on purpose, to show what happens when something fails.
We ask for two data types: RNA-Seq and Hi-C. There is one pass per data type for each species.

In [4]:
species = [
    {"taxid": 9685, "species": "Felis catus"},
    {"taxid": 9407, "species": "Rousettus aegyptiacus"},
    {"taxid": 59479, "species": "Rhinolophus ferrumequinum"},
    {"taxid": "cat", "species": "typo"},
]
data_types = ("RNA-Seq", "Hi-C")

## 2. Collect the runs

A species with no run for a data type is not a problem: it is kept with an empty list, because "no data" is useful information too.
`complete` is `False` only when ENA has more runs than we can read at once (5,000).

In [6]:
export = collect_raw_data(species, data_types)

print(len(export["species"]), "species / data type pairs,", len(export["problems"]), "problems")
for entry in export["species"]:
    print(f'{entry["species"]:26} {entry["data_type"]:8} ENA: {entry["total_runs"]:5} | useful: {entry["runs_useful"]:5} | chosen: {len(entry["runs"]):2} | complete: {entry["complete"]}')

6 species / data type pairs, 2 problems
Felis catus                RNA-Seq  ENA:  2000 | useful:  1541 | chosen: 10 | complete: True
Felis catus                Hi-C     ENA:    14 | useful:    14 | chosen:  4 | complete: True
Rousettus aegyptiacus      RNA-Seq  ENA:   361 | useful:   187 | chosen: 10 | complete: True
Rousettus aegyptiacus      Hi-C     ENA:     0 | useful:     0 | chosen:  0 | complete: True
Rhinolophus ferrumequinum  RNA-Seq  ENA:   202 | useful:   200 | chosen: 10 | complete: True
Rhinolophus ferrumequinum  Hi-C     ENA:     0 | useful:     0 | chosen:  0 | complete: True


## 3. How the runs were chosen

The file starts by saying which method and which settings were used, so anyone reading it knows how the runs were chosen.

In [7]:
print(json.dumps(export["selection"], indent=2))

{
  "method": "rules",
  "source": "ENA (also contains the runs of SRA and DDBJ)",
  "data_types": [
    "RNA-Seq",
    "Hi-C"
  ],
  "max_runs": 10,
  "min_bases": 1000000000,
  "excluded_sources": [
    "GENOMIC SINGLE CELL",
    "METAGENOMIC",
    "METATRANSCRIPTOMIC",
    "TRANSCRIPTOMIC SINGLE CELL"
  ]
}


## 4. The chosen runs of one species

Each chosen run keeps its metadata, its FASTQ links (with size and md5) and the reason why it was chosen.

In [8]:
bat = export["species"][2]   # Rousettus aegyptiacus, RNA-Seq
for run in bat["runs"]:
    print(run["run_accession"], run["tissue_type"], len(run["fastq"]), "files |", run["reason"])

SRR2914360 kidney 2 files | round 1: tissue 'kidney', from a study not used yet
SRR35797469 bone marrow 2 files | round 1: tissue 'bone marrow', from a study not used yet
SRR2914368 axillary lymph nodes 2 files | round 1: tissue 'axillary lymph nodes', from a study already used, but not for this tissue
SRR2914295 brain 2 files | round 1: tissue 'brain', from a study already used, but not for this tissue
SRR16761642 caudate nucleus of the basal ganglia 2 files | round 1: tissue 'caudate nucleus of the basal ganglia', from a study not used yet
SRR2914359 heart 2 files | round 1: tissue 'heart', from a study already used, but not for this tissue
SRR2914059 liver 2 files | round 1: tissue 'liver', from a study already used, but not for this tissue
SRR2913598 lung 2 files | round 1: tissue 'lung', from a study already used, but not for this tissue
SRR32671429 nasal epithelium 2 files | round 1: tissue 'nasal epithelium', from a study not used yet
SRR2914370 peripheral blood (pbmc) 2 files |

## 5. The problems

The species that failed are not lost: each one is listed with the reason, and the others still got their runs.

In [9]:
for problem in export["problems"]:
    print(problem["species"], problem["data_type"], "->", problem["message"])

typo RNA-Seq -> "cat" is not a valid taxid (it must be a number, e.g. 9685)
typo Hi-C -> "cat" is not a valid taxid (it must be a number, e.g. 9685)


## 6. Write the JSON file

This is the file a researcher or a pipeline would use.

In [10]:
write_export(export, "example_raw_data_export.json")
print(json.dumps(export, indent=2)[:3000])  # the beginning of the file

{
  "format": "genomexplore-raw-data",
  "format_version": "1.0",
  "created": "2026-11-09T18:53:28",
  "selection": {
    "method": "rules",
    "source": "ENA (also contains the runs of SRA and DDBJ)",
    "data_types": [
      "RNA-Seq",
      "Hi-C"
    ],
    "max_runs": 10,
    "min_bases": 1000000000,
    "excluded_sources": [
      "GENOMIC SINGLE CELL",
      "METAGENOMIC",
      "METATRANSCRIPTOMIC",
      "TRANSCRIPTOMIC SINGLE CELL"
    ]
  },
  "species": [
    {
      "taxid": "9685",
      "species": "Felis catus",
      "data_type": "RNA-Seq",
      "total_runs": 2000,
      "complete": true,
      "runs_useful": 1541,
      "runs": [
        {
          "run_accession": "SRR15199661",
          "study_accession": "PRJNA747757",
          "sample_accession": "SAMN20218752",
          "study_title": "Single cell atlas for mammals, reptiles and birds",
          "sample_title": "Model organism or animal sample from Felis catus",
          "layout": "PAIRED",
          "li